# RubaiSTT v2 — телефондан ўзбекча овозни синаш

**Бу Google Colab ноутбуги фақат интерактив синов учун.** У Telegram ботга уланмайди, орқа фонда сервер очмайди ва доимий касса ҳисоби учун ишлатилмайди.

1. Телефон браузерида Colab'га Google аккаунтингиз билан киринг. `Runtime → Change runtime type` бўлимида **T4 GPU** бор бўлса танланг (бепул GPU ҳар доим ҳам берилмайди).
2. `Runtime → Run all` босинг ёки катакчаларни тепадан пастга кетма-кет ишга туширинг.
3. Охирги катакчада Telegram'дан сақланган 30 сониягача бўлган **тест** овозли файлни танланг (`.ogg`, `.oga`, `.mp3`, `.wav`).
4. Пастда модел таниб олган ўзбекча матнни кўрасиз. Сумма, валюта ва «олдим/бердим»ни алоҳида текширинг.

Модел: https://huggingface.co/islomov/rubaistt_v2_medium (тахминан 3,06 ГБ юклаб олинади). Биринчи ишга тушиш узоқ давом этиши мумкин, GPU берилмаса CPU жуда секин ишлаши мумкин. Colab Google хизмати: махфий ишлаб турган касса аудиоларини эмас, фақат синов овозини юкланг.


In [ ]:
import subprocess, shutil
if not shutil.which('ffmpeg'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', 'transformers>=4.48,<5', 'soundfile>=0.12', 'accelerate>=0.30', 'safetensors>=0.4'], check=True)
print('Кутубхоналар тайёр. Кейинги катакчани ишга туширинг.')


In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
MODEL_ID = 'islomov/rubaistt_v2_medium'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32
print('Қурилма:', device)
if device == 'cpu': print('GPU мавжуд эмас: овозни таниш секин бўлиши мумкин.')
print('RubaiSTT v2 модели юкланмоқда (~3.06 GB)...')
processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=dtype, low_cpu_mem_usage=True).to(device).eval()
print('Модел тайёр. Охирги катакчада тест аудио юкланг.')


In [ ]:
import io, gc, subprocess, soundfile as sf
from google.colab import files
print('Сумма ва валюталарни айтган 30 сониягача бўлган ТЕСТ овозни танланг.')
uploaded = files.upload()
if len(uploaded) != 1: raise ValueError('Фақат битта тест аудиони танланг.')
name, data = next(iter(uploaded.items()))
if not data or len(data) > 10_000_000: raise ValueError('Файл бўш ёки 10 МБдан катта.')
result = subprocess.run(['ffmpeg', '-nostdin', '-v', 'error', '-i', 'pipe:0', '-t', '31', '-ar', '16000', '-ac', '1', '-f', 'wav', 'pipe:1'], input=data, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=35, check=True)
audio, sample_rate = sf.read(io.BytesIO(result.stdout), dtype='float32')
if sample_rate != 16000 or not 0 < len(audio) <= 30 * 16000: raise ValueError('Аудио 30 сониядан ошмаслиги керак.')
features = processor(audio, sampling_rate=16000, return_tensors='pt').input_features.to(device=device, dtype=dtype)
with torch.inference_mode():
    tokens = model.generate(features, language='uz', task='transcribe', max_new_tokens=128)
recognized = processor.batch_decode(tokens, skip_special_tokens=True)[0].strip()
print('\nRubaiSTT v2 таниб олган матн:\n', recognized or '(Бўш натижа)')
# Google Colab runtime файлни қайта ишга туширгунча сақлаб қолмаслиги учун шу ерда ўчирилади.
import os
if os.path.isfile(name): os.remove(name)
uploaded.clear(); del data, audio, result, features, tokens
gc.collect()
